# TrafficFlowBench pipeline

Writes the pipeline files, then runs Tasks 1, 2 and 4 on the attached competition data and builds `outputs/submission.csv`. Task 3 is scored on the Task 1 rows.

In [ ]:
import os
for d in ('src', 'official/src', 'official/config'):
    os.makedirs(f'/kaggle/working/tfb/{d}', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/tfb/src/common.py
"""Shared helpers: release paths, dense per-panel caches, network parameters, local scoring.

A cache holds one panel and one split as arrays shaped [day, slot(288), link]:
  speed, flow, occ   values as released in the masked layer (NaN where blanked)
  elig               is_score_eligible
  target             Task 1 target cells, read from the published template
  regime             per-day mask regime index (0, 1, 2 = R1, R2, R3)
  true_speed/flow    train only: the unmasked layer, which is the Task 1 truth
"""
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path(__file__).resolve().parent.parent
DEFAULT_RELEASE = ROOT / "data" / "kaggle_public"
CACHE = Path(os.environ.get("TFB_CACHE", ROOT / "cache"))
OUTPUTS = ROOT / "outputs"
SPLITS = ("train", "validation", "private")
REGIMES = ("R1", "R2", "R3")
SLOTS = 288

# Task 1 scoring constants, from official/src/task1/baseline_task1_historical_mean.py
SPEED_NORM, FLOW_NORM, SPEED_W, FLOW_W = 25.0, 600.0, 0.54, 0.46


def panels(release: Path) -> list[str]:
    manifest = release / "config" / "corridors.json"
    if not manifest.exists():
        manifest = ROOT / "official" / "config" / "corridors.json"
    found = {p.name for p in (release / "corridors").iterdir() if p.is_dir()}
    listed = [p["corridor_id"] for p in json.loads(manifest.read_text())["panels"]]
    return [p for p in listed if p in found] + sorted(found - set(listed))


def slot_of(ts: pd.Series) -> np.ndarray:
    """5-minute slot of a canonical text timestamp, YYYY-MM-DDTHH:MM:SSZ."""
    s = ts.astype(str)
    return (s.str.slice(11, 13).astype(int) * 12 + s.str.slice(14, 16).astype(int) // 5).to_numpy()


def read_parquet_dir(root: Path, columns: list[str]) -> pd.DataFrame:
    files = sorted(root.glob("**/*.parquet"))
    if not files:
        return pd.DataFrame(columns=columns)
    parts = []
    for f in files:
        have = set(pq.read_schema(f).names)
        parts.append(pq.read_table(f, columns=[c for c in columns if c in have]).to_pandas())
    return pd.concat(parts, ignore_index=True)


def link_order(panel_dir: Path, frame: pd.DataFrame) -> list[str]:
    """Mainline links in corridor order: topology order_index, else milepost, else id."""
    seen = set(frame.link_id.astype(str))
    topo = panel_dir / "network" / "lwr_mainline_topology.csv"
    if topo.exists():
        t = pd.read_csv(topo, dtype=str)
        if "order_index" in t.columns:
            t["order_index"] = pd.to_numeric(t.order_index, errors="coerce")
            order = [x for x in t.sort_values("order_index").link_id.drop_duplicates() if x in seen]
            return order + sorted(seen - set(order))
    if "milepost" in frame.columns:
        m = frame[["link_id", "milepost"]].drop_duplicates("link_id").copy()
        m["link_id"] = m.link_id.astype(str)
        m["milepost"] = pd.to_numeric(m.milepost, errors="coerce")
        return m.sort_values(["milepost", "link_id"]).link_id.tolist()
    return sorted(seen)


MAIN_COLS = ["date", "timestamp", "station_id", "link_id", "speed_kmh", "flow_vph", "occupancy",
             "pct_observed", "is_score_eligible", "mask_regime", "milepost"]


def build_cache(release: Path, panel: str, split: str, links: list[str] | None = None) -> dict:
    """Dense arrays for one panel/split, written to cache/<panel>_<split>.npz."""
    pdir = release / "corridors" / panel
    m = read_parquet_dir(pdir / split / "mainline_states_masked", MAIN_COLS)
    if m.empty:
        raise FileNotFoundError(f"{panel}/{split}: no mainline_states_masked partitions under {pdir / split}")
    m["link_id"] = m.link_id.astype(str)
    m["station_id"] = m.station_id.astype(str)
    if "date" not in m.columns:
        m["date"] = m.timestamp.astype(str).str.slice(0, 10)
    if links is None:
        links = link_order(pdir, m)
    dates = sorted(m.date.astype(str).unique())
    lmap = {x: i for i, x in enumerate(links)}
    dmap = {x: i for i, x in enumerate(dates)}
    m = m[m.link_id.isin(lmap)]
    d, s, l = m.date.astype(str).map(dmap).to_numpy(), slot_of(m.timestamp), m.link_id.map(lmap).to_numpy()

    def dense(values, fill=np.nan, dtype=np.float32):
        a = np.full((len(dates), SLOTS, len(links)), fill, dtype=dtype)
        a[d, s, l] = values
        return a

    out = {
        "speed": dense(pd.to_numeric(m.speed_kmh, errors="coerce").to_numpy(np.float32)),
        "flow": dense(pd.to_numeric(m.flow_vph, errors="coerce").to_numpy(np.float32)),
        "elig": dense(m.is_score_eligible.fillna(False).astype(bool).to_numpy(), False, bool),
    }
    if "occupancy" in m.columns:
        out["occ"] = dense(pd.to_numeric(m.occupancy, errors="coerce").to_numpy(np.float32))
    reg = m.groupby(m.date.astype(str)).mask_regime.first().reindex(dates).astype(str)
    out["regime"] = np.array([REGIMES.index(r) if r in REGIMES else 0 for r in reg], dtype=np.int8)
    st = m.drop_duplicates("link_id").set_index("link_id").station_id.reindex(links).fillna("").astype(str)

    tpl_path = release / "task1" / panel / split / "sample_submission_state.csv"
    target = np.zeros(out["speed"].shape, dtype=bool)
    if tpl_path.exists():
        tpl = pd.read_csv(tpl_path, usecols=["timestamp", "link_id"], dtype=str)
        tpl = tpl[tpl.link_id.isin(lmap) & tpl.timestamp.str.slice(0, 10).isin(dmap)]
        target[tpl.timestamp.str.slice(0, 10).map(dmap).to_numpy(), slot_of(tpl.timestamp),
               tpl.link_id.map(lmap).to_numpy()] = True
    out["target"] = target

    if split == "train":
        f = read_parquet_dir(pdir / split / "mainline_states", ["date", "timestamp", "link_id", "speed_kmh", "flow_vph"])
        f["link_id"] = f.link_id.astype(str)
        if "date" not in f.columns:
            f["date"] = f.timestamp.astype(str).str.slice(0, 10)
        f = f[f.link_id.isin(lmap) & f.date.astype(str).isin(dmap)]
        fd_, fs, fl = f.date.astype(str).map(dmap).to_numpy(), slot_of(f.timestamp), f.link_id.map(lmap).to_numpy()
        for col, name in (("speed_kmh", "true_speed"), ("flow_vph", "true_flow")):
            a = np.full(out["speed"].shape, np.nan, dtype=np.float32)
            a[fd_, fs, fl] = pd.to_numeric(f[col], errors="coerce").to_numpy(np.float32)
            out[name] = a

    CACHE.mkdir(parents=True, exist_ok=True)
    np.savez(CACHE / f"{panel}_{split}.npz", dates=np.array(dates, dtype=str), links=np.array(links, dtype=str),
             stations=st.to_numpy().astype(str), **out)
    return load_cache(panel, split)


def load_cache(panel: str, split: str) -> dict:
    z = np.load(CACHE / f"{panel}_{split}.npz", allow_pickle=False)
    return {k: z[k] for k in z.files}


def cached(release: Path, panel: str, split: str, rebuild: bool = False) -> dict:
    path = CACHE / f"{panel}_{split}.npz"
    if path.exists() and not rebuild:
        return load_cache(panel, split)
    links = None
    if split != "train":
        links = list(cached(release, panel, "train", rebuild=False)["links"])
    return build_cache(release, panel, split, links)


def network(release: Path, panel: str, links) -> pd.DataFrame:
    """Per-link length, lanes, free speed, capacity, jam density and queue cutoff, in cache link order.

    fd_parameters.csv wins over links.csv where both carry a value, as in the official Task 3 scorer.
    """
    ndir = release / "corridors" / panel / "network"
    links = [str(x) for x in links]
    net = pd.DataFrame(index=pd.Index(links, name="link_id"))
    if (ndir / "links.csv").exists():
        lk = pd.read_csv(ndir / "links.csv", dtype={"link_id": str}).drop_duplicates("link_id").set_index("link_id")
        for c in ("length_km", "lanes", "free_speed_kmh", "capacity_vph"):
            if c in lk.columns:
                net[c] = pd.to_numeric(lk[c], errors="coerce").reindex(links)
    if (ndir / "fd_parameters.csv").exists():
        fd = pd.read_csv(ndir / "fd_parameters.csv", dtype={"link_id": str, "station_id": str})
        fd = fd.groupby("link_id").mean(numeric_only=True).reindex(links)
        for c in ("length_km", "lanes", "free_speed_kmh", "capacity_vph", "k_jam", "v_cut"):
            if c in fd.columns:
                net[c] = fd[c].combine_first(net[c]) if c in net.columns else fd[c]
    defaults = {"length_km": 0.5, "lanes": 4.0, "free_speed_kmh": 105.0}
    for c, v in defaults.items():
        net[c] = pd.to_numeric(net.get(c), errors="coerce").fillna(v) if c in net.columns else v
    net.loc[(net.lanes < 1) | (net.lanes > 16), "lanes"] = 4.0
    if "capacity_vph" not in net.columns:
        net["capacity_vph"] = np.nan
    net["capacity_vph"] = net.capacity_vph.fillna(2000.0 * net.lanes)
    if "k_jam" not in net.columns:
        net["k_jam"] = np.nan
    net["k_jam"] = net.k_jam.fillna(110.0 * net.lanes)
    if "v_cut" not in net.columns:
        net["v_cut"] = np.nan
    # The Task 2 queue label: speed at or below 0.60 x free-flow speed.
    net["v_cut"] = net.v_cut.fillna(0.60 * net.free_speed_kmh)
    return net


def weekday_of(dates) -> np.ndarray:
    return pd.to_datetime(pd.Series(list(dates))).dt.dayofweek.to_numpy()


def score_state(pv: np.ndarray, pq_: np.ndarray, z: dict, lanes: np.ndarray, days=None) -> dict:
    """Task 1 score on a train cache, same formula as official/src/task1/score_task1.py."""
    days = np.arange(len(z["dates"])) if days is None else np.asarray(days)
    tv, tq = z["true_speed"], z["true_flow"]
    out, parts = {}, []
    for r, name in enumerate(REGIMES):
        dd = days[z["regime"][days] == r]
        if len(dd) == 0:
            continue
        m = z["target"][dd] & z["elig"][dd] & np.isfinite(tv[dd]) & np.isfinite(tq[dd])
        ev = np.nan_to_num(pv[dd], nan=0.0)[m] - tv[dd][m]
        eq = ((np.nan_to_num(pq_[dd], nan=0.0) - tq[dd]) / lanes)[m]
        rv, rq = float(np.sqrt(np.mean(ev ** 2))), float(np.sqrt(np.mean(eq ** 2)))
        s = SPEED_W * max(0.0, 1 - rv / SPEED_NORM) + FLOW_W * max(0.0, 1 - rq / FLOW_NORM)
        out[name] = {"rmse_speed_kmh": rv, "rmse_flow_vph_per_lane": rq, "S_state": s, "n": int(m.sum())}
        parts.append(s)
    out["S_state"] = float(np.mean(parts)) if parts else 0.0
    return out


In [ ]:
%%writefile /kaggle/working/tfb/src/task1_state.py
"""Task 1: reconstruct the masked mainline speed and flow cells.

Every candidate method is scored on the train split, where the truth is released, and the best one is chosen
per panel and per channel (speed, flow) before it is applied to validation and private.

Candidates:
  interp           linear interpolation in time on the same link, between the nearest released cells
  profile+resid    weekday x time-of-day profile plus the time-interpolated residual from it, which decays
                   back to the profile as the gap grows (tau = decay length in 5-minute slots)
  ...+nbr          adds a same-instant correction from neighbouring links' residuals, fitted by least squares
                   on released cells, where no labels are needed

Profiles are built from released (masked) cells only, so scoring on train never sees a target's truth.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from common import REGIMES, SLOTS, score_state, weekday_of

TAUS = (6.0, 24.0, np.inf)
NBR_OFFSETS = (-2, -1, 1, 2)


def time_interp(x: np.ndarray, loo: bool = False):
    """Linear interpolation along axis 0 of x [T, L] from the nearest finite cells before and after.

    With loo=True the cell itself is excluded, so a released cell gets the estimate it would have had as a
    target. Returns (estimate, gap to previous, gap to next); a gap is 10**6 where there is no neighbour.
    """
    T, L = x.shape
    ok = np.isfinite(x)
    idx = np.arange(T)[:, None]
    prev = np.maximum.accumulate(np.where(ok, idx, -1), axis=0)
    nxt = np.minimum.accumulate(np.where(ok, idx, 2 * T)[::-1], axis=0)[::-1]
    if loo:
        prev = np.vstack([np.full((1, L), -1), prev[:-1]])
        nxt = np.vstack([nxt[1:], np.full((1, L), 2 * T)])
    hp, hn = prev >= 0, nxt < T
    cols = np.arange(L)[None, :]
    xp = np.where(hp, x[np.clip(prev, 0, T - 1), cols], np.nan)
    xn = np.where(hn, x[np.clip(nxt, 0, T - 1), cols], np.nan)
    gp = np.where(hp, idx - prev, 10 ** 6).astype(np.float64)
    gn = np.where(hn, nxt - idx, 10 ** 6).astype(np.float64)
    w = np.where(hp & hn, gn / np.maximum(gp + gn, 1), np.where(hp, 1.0, 0.0))
    est = np.where(hp & hn, w * xp + (1 - w) * xn, np.where(hp, xp, xn))
    return est, gp, gn


def profile(arrays: list[np.ndarray], weekdays: list[np.ndarray]) -> np.ndarray:
    """Mean by (weekday, slot, link) over released cells of several splits; link mean where a slot is empty."""
    L = arrays[0].shape[2]
    s, n = np.zeros((7, SLOTS, L)), np.zeros((7, SLOTS, L))
    for a, wd in zip(arrays, weekdays):
        for k in range(7):
            sel = a[wd == k]
            s[k] += np.nansum(sel, axis=0)
            n[k] += np.isfinite(sel).sum(axis=0)
    link_mean = s.sum((0, 1)) / np.maximum(n.sum((0, 1)), 1)
    return np.where(n > 0, s / np.maximum(n, 1), link_mean[None, None, :])


def neighbour_feature(R: np.ndarray) -> np.ndarray:
    """Mean of R [T, L] over links l-2..l+2 (excluding l) at the same instant, NaN-aware."""
    num, den = np.zeros_like(R), np.zeros_like(R)
    for k in NBR_OFFSETS:
        sh = np.full_like(R, np.nan)
        if k > 0:
            sh[:, :-k] = R[:, k:]
        else:
            sh[:, -k:] = R[:, :k]
        w = 1.0 / abs(k)
        num += w * np.nan_to_num(sh)
        den += w * np.isfinite(sh)
    return np.where(den > 0, num / np.maximum(den, 1e-9), 0.0)


def candidate(x: np.ndarray, P: np.ndarray, tau: float | None, nbr: bool, relative: bool):
    """One candidate estimate for every cell of x [T, L] (flattened days), with profile P aligned to x."""
    def base(loo):
        if tau is None:
            est, gp, gn = time_interp(x, loo)
            return np.where(np.isfinite(est), est, P), gp, gn
        r = (x / np.maximum(P, 1e-6) - 1.0) if relative else (x - P)
        er, gp, gn = time_interp(r, loo)
        decay = np.exp(-np.minimum(gp, gn) / tau) if np.isfinite(tau) else 1.0
        er = np.nan_to_num(er) * decay
        return (P * (1 + er) if relative else P + er), gp, gn

    est, _, _ = base(False)
    if not nbr:
        return est
    loo, gp, gn = base(True)
    R = (x / np.maximum(loo, 1e-6) - 1.0) if relative else (x - loo)
    R = np.where(np.isfinite(R) & (np.maximum(gp, gn) <= 3), R, np.nan)
    lim = np.nanstd(R) * 5 if np.isfinite(R).any() else 0.0
    R = np.clip(R, -lim, lim)
    F = neighbour_feature(R)
    fit = np.isfinite(R) & (F != 0)
    if fit.sum() < 1000:
        return est
    beta = float((F[fit] * R[fit]).sum() / max((F[fit] ** 2).sum(), 1e-12))
    beta = float(np.clip(beta, 0.0, 1.0))
    return est * (1 + beta * F) if relative else est + beta * F


def configs():
    for tau in (None,) + TAUS:
        for nbr in (False, True):
            name = ("interp" if tau is None else f"profile+resid(tau={tau:g})") + ("+nbr" if nbr else "")
            yield name, tau, nbr


def _flat(a):
    return a.reshape(-1, a.shape[-1]).astype(np.float64)


def predict(z: dict, P_v: np.ndarray, P_q: np.ndarray, cfg_v, cfg_q, vmax: np.ndarray):
    D, S, L = z["speed"].shape
    wd = weekday_of(z["dates"])
    Pv, Pq = _flat(P_v[wd]), _flat(P_q[wd])
    pv = candidate(_flat(z["speed"]), Pv, cfg_v[0], cfg_v[1], relative=False)
    pq = candidate(_flat(z["flow"]), Pq, cfg_q[0], cfg_q[1], relative=True)
    pv = np.clip(pv, 1.0, vmax[None, :]).reshape(D, S, L)
    pq = np.clip(pq, 0.0, None).reshape(D, S, L)
    return pv.astype(np.float32), pq.astype(np.float32)


def select(z_train: dict, P_v, P_q, lanes: np.ndarray, vmax: np.ndarray, log=print) -> dict:
    """Score every candidate on train, separately for speed and flow, and keep the best of each."""
    D, S, L = z_train["speed"].shape
    wd = weekday_of(z_train["dates"])
    Pv, Pq = _flat(P_v[wd]), _flat(P_q[wd])
    xv, xq = _flat(z_train["speed"]), _flat(z_train["flow"])
    m = z_train["target"] & z_train["elig"] & np.isfinite(z_train["true_speed"]) & np.isfinite(z_train["true_flow"])
    m = m.reshape(D * S, L)
    tv, tq = _flat(z_train["true_speed"]), _flat(z_train["true_flow"])
    lane = np.broadcast_to(lanes[None, :], m.shape)
    results = []
    for name, tau, nbr in configs():
        ev = np.clip(candidate(xv, Pv, tau, nbr, False), 1.0, vmax[None, :])
        eq = np.clip(candidate(xq, Pq, tau, nbr, True), 0.0, None)
        rv = float(np.sqrt(np.mean((ev[m] - tv[m]) ** 2)))
        rq = float(np.sqrt(np.mean(((eq[m] - tq[m]) / lane[m]) ** 2)))
        results.append({"method": name, "tau": tau, "nbr": nbr, "rmse_speed": rv, "rmse_flow_per_lane": rq})
        log(f"    {name:32s} speed RMSE {rv:6.3f} km/h   flow RMSE {rq:7.2f} veh/h/lane")
    best_v = min(results, key=lambda r: r["rmse_speed"])
    best_q = min(results, key=lambda r: r["rmse_flow_per_lane"])
    pv, pq = predict(z_train, P_v, P_q, (best_v["tau"], best_v["nbr"]), (best_q["tau"], best_q["nbr"]), vmax)
    score = score_state(pv, pq, z_train, lanes)
    hist = score_state(np.asarray(P_v[wd], np.float32), np.asarray(P_q[wd], np.float32), z_train, lanes)
    return {"speed_method": best_v, "flow_method": best_q, "candidates": results,
            "train_score": score, "train_score_profile_only": hist}


def submission_rows(panel: str, template: pd.DataFrame, z: dict, pv: np.ndarray, pq: np.ndarray) -> pd.DataFrame:
    """Fill the published template rows, keeping its keys verbatim."""
    from common import slot_of
    dmap = {x: i for i, x in enumerate(z["dates"].astype(str))}
    lmap = {x: i for i, x in enumerate(z["links"].astype(str))}
    t = template.copy()
    d = t.timestamp.str.slice(0, 10).map(dmap)
    l = t.link_id.map(lmap)
    ok = d.notna() & l.notna()
    d, l, s = d.fillna(0).astype(int).to_numpy(), l.fillna(0).astype(int).to_numpy(), slot_of(t.timestamp)
    t["panel"] = panel
    if "mask_regime" not in t.columns or t.mask_regime.isna().any():
        t["mask_regime"] = np.asarray(REGIMES)[z["regime"][d]]
    t["speed_kmh"] = np.where(ok, np.round(pv[d, s, l], 3), np.nan)
    t["flow_vph"] = np.where(ok, np.round(pq[d, s, l], 2), np.nan)
    return t[["panel", "timestamp", "station_id", "link_id", "mask_regime", "speed_kmh", "flow_vph"]]


In [ ]:
%%writefile /kaggle/working/tfb/src/task2_queue.py
"""Task 2: predict which links are queued at T+5 .. T+30 from one hour of history.

A link is queued when its speed is at or below v_cut (0.60 x free-flow speed). The leaderboard scores
space-time IoU per window, averaged over windows, with queue_onset and queue_ongoing weighted equally.

Training windows are generated from every train day, where the unmasked layer gives the labels, using the
same conditions the organizers select on. Onset windows are reweighted so that the step at which the queue
first appears matches the shipped train windows. Features only use timestamps at or before T.

One LightGBM model per condition is evaluated with day-grouped cross-validation, on generated windows and on
the organizers' own train windows, against the persistence baseline. The cell selection rule (a fixed
probability threshold, or the set that maximizes expected IoU) is chosen by the same cross-validation.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from common import SLOTS, weekday_of

HIST = 12          # history steps T-60 .. T-5, plus the cell at T
H = 6              # horizon steps T+5 .. T+30
CONDITIONS = ("queue_onset", "queue_ongoing")
NBR = (-5, -3, -2, -1, 1, 2, 3, 5)
FEATURES = (
    [f"r_lag{k}" for k in range(HIST + 1)]
    + ["r_last", "r_min", "r_mean3", "r_slope3", "q_lane_last", "q_lane_slope3"]
    + [f"nbr{k:+d}_r_last" for k in NBR] + [f"nbr{k:+d}_r_min" for k in (-2, -1, 1, 2)]
    + ["dist_q_down", "dist_q_up", "n_queued_now", "frac_queued_hist",
       "freq_now", "freq_h", "freq_h_max_nbr",
       "h", "tod", "weekend", "pos", "v_cut", "lanes"]
)


# ---------------------------------------------------------------- features

def _ffill(a):
    idx = np.where(np.isfinite(a), np.arange(a.shape[0])[:, None], 0)
    np.maximum.accumulate(idx, axis=0, out=idx)
    out = a[idx, np.arange(a.shape[1])[None, :]]
    return out


def _shift(x, k):
    """x[..., l+k], NaN outside the corridor."""
    out = np.full_like(x, np.nan)
    if k > 0:
        out[..., :-k] = x[..., k:]
    elif k < 0:
        out[..., -k:] = x[..., :k]
    return out


def _dist(flag, direction):
    L = flag.shape[0]
    out = np.full(L, 99.0)
    rng = range(L - 2, -1, -1) if direction > 0 else range(1, L)
    for i in rng:
        j = i + direction
        out[i] = 1.0 if flag[j] else min(out[j] + 1, 99.0)
    return out


def window_features(hv, hq, vcut, lanes, freq_we, s0, weekend):
    """Feature matrix [H * L, F] for one window.

    hv, hq: [HIST + 1, L] speed and flow at T-60 .. T (row HIST is T), NaN where unavailable.
    freq_we: [SLOTS, L] train queue frequency for this window's day type.
    """
    L = hv.shape[1]
    r = hv / vcut[None, :]
    rf = _ffill(r)
    r_last = rf[-1]
    with np.errstate(all="ignore"):
        r_min = np.nanmin(r, axis=0)
        r_mean3 = np.nanmean(r[-3:], axis=0)
        ql = hq / lanes[None, :]
        qf = _ffill(ql)
    r_slope3 = rf[-1] - rf[-4]
    queued_now = np.nan_to_num(r_last, nan=9.0) <= 1.0
    per_link = [r[::-1].T, r_last, r_min, r_mean3, r_slope3, qf[-1], qf[-1] - qf[-4]]
    per_link += [_shift(r_last, k) for k in NBR] + [_shift(r_min, k) for k in (-2, -1, 1, 2)]
    per_link += [_dist(queued_now, +1), _dist(queued_now, -1),
                 np.full(L, queued_now.sum(), float),
                 np.full(L, np.nanmean(np.nan_to_num(r, nan=9.0) <= 1.0), float),
                 freq_we[min(s0, SLOTS - 1)]]
    base = np.column_stack([p if p.ndim == 2 else p[:, None] for p in per_link])
    rows = []
    for h in range(1, H + 1):
        s = min(s0 + h, SLOTS - 1)
        fh = freq_we[s]
        fnb = np.nanmax(np.stack([_shift(fh, k) for k in (-2, -1, 1, 2)]), axis=0)
        extra = np.column_stack([fh, fnb, np.full(L, h), np.full(L, s0), np.full(L, weekend),
                                 np.arange(L) / max(L - 1, 1), vcut, lanes])
        rows.append(np.hstack([base, extra]))
    return np.vstack(rows).astype(np.float32)


# ---------------------------------------------------------------- queue frequency from train

def queue_frequency(z_train, vcut):
    tv = z_train["true_speed"]
    ok = np.isfinite(tv) & z_train["elig"]
    q = ok & (tv <= vcut[None, None, :])
    we = (weekday_of(z_train["dates"]) >= 5).astype(int)
    S = np.stack([q[we == k].sum(0) for k in (0, 1)]).astype(np.float64)
    N = np.stack([ok[we == k].sum(0) for k in (0, 1)]).astype(np.float64)
    return q, ok, we, S, N


def freq_for(S, N, we_k, own_q=None, own_ok=None):
    s, n = S[we_k].copy(), N[we_k].copy()
    if own_q is not None:
        s -= own_q
        n -= own_ok
    return s / np.maximum(n, 1)


# ---------------------------------------------------------------- windows

def _labels_and_conditions(tv, el, vcut):
    """Per (slot, link) queue truth for one day, with NaN where not scoreable."""
    lab = np.where(np.isfinite(tv) & el, (tv <= vcut[None, :]).astype(float), np.nan)
    return lab


def generate_windows(z_train, vcut, rng, max_per_condition):
    """Origins on train days that satisfy the organizers' condition definitions."""
    tv = z_train["true_speed"]
    el = z_train["elig"]
    out = {c: [] for c in CONDITIONS}
    for d in range(len(z_train["dates"])):
        lab = _labels_and_conditions(tv[d], el[d], vcut)
        q = np.nan_to_num(lab) > 0
        cover = np.isfinite(lab)
        for s0 in range(HIST, SLOTS - H):
            hist, fut = q[s0 - HIST:s0 + 1], q[s0 + 1:s0 + 1 + H]
            if not fut.any() or cover[s0 - HIST:s0].mean() < 0.7:
                continue
            if not hist.any():
                first = int(np.argmax(fut.any(axis=1))) + 1
                out["queue_onset"].append((d, s0, first))
            elif q[s0].any() or q[s0 - 1].any():
                pers = np.repeat((q[s0] | q[s0 - 1])[None, :], H, 0)
                m = cover[s0 + 1:s0 + 1 + H]
                inter, union = (pers & fut & m).sum(), ((pers | fut) & m).sum()
                if union and inter / union <= 0.9:
                    out["queue_ongoing"].append((d, s0, 0))
    for c in CONDITIONS:
        w = out[c]
        if len(w) > max_per_condition:
            w = [w[i] for i in sorted(rng.choice(len(w), max_per_condition, replace=False))]
        out[c] = w
    return out


def read_task2(release: Path, panel: str, split: str):
    base = release / "task2" / panel / split
    if not (base / "window_index.csv").exists():
        return None
    wi = pd.read_csv(base / "window_index.csv", dtype=str)
    origin_col = next(c for c in ("forecast_origin", "origin", "t0", "forecast_origin_utc") if c in wi.columns)
    wi["forecast_origin"] = wi[origin_col]
    if "date" not in wi.columns:
        wi["date"] = wi.forecast_origin.str.slice(0, 10)
    hist = pd.read_parquet(base / "window_history.parquet")
    hist["window_id"] = hist.window_id.astype(str)
    hist["link_id"] = hist.link_id.astype(str)
    tpl = pd.read_csv(base / "sample_submission_queue.csv", dtype=str) \
        if (base / "sample_submission_queue.csv").exists() else None
    return wi, hist, tpl


def released_window_arrays(w, hist_w, z, links_index):
    """History speed/flow [HIST+1, L] for a published window: T-60 .. T-5 from window_history, T from the
    masked layer of the same split (released data stamped at T)."""
    L = len(links_index)
    hv = np.full((HIST + 1, L), np.nan)
    hq = np.full((HIST + 1, L), np.nan)
    origin = pd.Timestamp(w.forecast_origin)
    ts = pd.to_datetime(hist_w.timestamp, utc=True)
    step = ((ts - origin) / pd.Timedelta(minutes=5)).round().astype(int).to_numpy() + HIST
    li = hist_w.link_id.map(links_index)
    ok = (step >= 0) & (step < HIST) & li.notna().to_numpy()
    li = li.fillna(0).astype(int).to_numpy()
    hv[step[ok], li[ok]] = pd.to_numeric(hist_w.speed_kmh, errors="coerce").to_numpy()[ok]
    if "flow_vph" in hist_w.columns:
        hq[step[ok], li[ok]] = pd.to_numeric(hist_w.flow_vph, errors="coerce").to_numpy()[ok]
    s0 = origin.hour * 12 + origin.minute // 5
    dmap = {x: i for i, x in enumerate(z["dates"].astype(str))}
    d = dmap.get(str(w.date))
    if d is not None:
        hv[HIST] = z["speed"][d, s0]
        hq[HIST] = z["flow"][d, s0]
        if "flow_vph" not in hist_w.columns:
            lo = max(s0 - HIST, 0)
            hq[HIST - (s0 - lo):HIST] = z["flow"][d, lo:s0]
    return hv, hq, s0, d


def persistence(hv, vcut):
    """The official baseline: repeat the queue state at the last history step through the horizon."""
    last = _ffill(hv)[HIST - 1]
    return np.repeat((np.nan_to_num(last, nan=9e9) <= vcut)[None, :], H, 0).astype(float)


# ---------------------------------------------------------------- selection rules

def select_cells(p, rule):
    """Binary prediction for one window from probabilities p [H*L]."""
    if rule[0] == "threshold":
        pred = p >= rule[1]
        if not pred.any():
            pred = p >= 0.5 * p.max()
        return pred
    order = np.argsort(-p)
    cs = np.cumsum(p[order])
    k = np.arange(1, len(p) + 1)
    exp_iou = cs / (k + p.sum() - cs)
    best = int(np.argmax(exp_iou)) + 1
    pred = np.zeros(len(p), bool)
    pred[order[:best]] = True
    return pred


def iou(pred, y):
    m = np.isfinite(y)
    yt = y[m] > 0
    pr = pred[m]
    union = (pr | yt).sum()
    return 1.0 if union == 0 else float((pr & yt).sum() / union)


RULES = [("threshold", t) for t in (0.2, 0.3, 0.4, 0.5, 0.6)] + [("expected_iou", None)]


# ---------------------------------------------------------------- tables, training, prediction

def panel_tables(release: Path, panel: str, z_tr: dict, scored: dict, net: pd.DataFrame, rng, max_windows: int):
    """Rows for one panel: generated train windows, shipped train windows, and the scored splits' windows."""
    vcut, lanes = net.v_cut.to_numpy(float), net.lanes.to_numpy(float)
    lmap = {x: i for i, x in enumerate(z_tr["links"].astype(str))}
    L = len(lmap)
    q, ok, we, S, N = queue_frequency(z_tr, vcut)
    wd = weekday_of(z_tr["dates"])
    rows = []

    def add(kind, cond, split, date, wid, X, y, first, meta=None):
        rows.append(dict(kind=kind, condition=cond, split=split, panel=panel, date=date, window_id=wid,
                         X=X, y=y, first=first, meta=meta))

    gen = generate_windows(z_tr, vcut, rng, max_windows)
    for cond, ws in gen.items():
        for d, s0, first in ws:
            hv = z_tr["true_speed"][d, s0 - HIST:s0 + 1].astype(float).copy()
            hq = z_tr["true_flow"][d, s0 - HIST:s0 + 1].astype(float).copy()
            hv[HIST], hq[HIST] = z_tr["speed"][d, s0], z_tr["flow"][d, s0]
            freq = freq_for(S, N, we[d], q[d], ok[d])
            X = window_features(hv, hq, vcut, lanes, freq, s0, float(wd[d] >= 5))
            lab = _labels_and_conditions(z_tr["true_speed"][d], z_tr["elig"][d], vcut)[s0 + 1:s0 + 1 + H]
            add("generated", cond, "train", str(z_tr["dates"][d]), f"gen_{panel}_{d}_{s0}", X, lab.reshape(-1), first)

    for split, z in [("train", z_tr)] + list(scored.items()):
        t2 = read_task2(release, panel, split)
        if t2 is None:
            continue
        wi, hist, tpl = t2
        by_window = dict(tuple(hist.groupby("window_id")))
        for w in wi.itertuples(index=False):
            hw = by_window.get(str(w.window_id))
            if hw is None:
                continue
            hv, hq, s0, d = released_window_arrays(w, hw, z, lmap)
            if split == "train" and d is not None:
                freq = freq_for(S, N, we[d], q[d], ok[d])
                lab = _labels_and_conditions(z_tr["true_speed"][d], z_tr["elig"][d], vcut)
                lab = np.full((H, L), np.nan) if s0 + H >= SLOTS else lab[s0 + 1:s0 + 1 + H]
            else:
                weekend = pd.Timestamp(w.date).dayofweek >= 5
                freq = freq_for(S, N, int(weekend))
                lab = np.full((H, L), np.nan)
            weekend = float(pd.Timestamp(w.date).dayofweek >= 5)
            X = window_features(hv, hq, vcut, lanes, freq, s0, weekend)
            y = lab.reshape(-1)
            first = int(np.argmax(np.nan_to_num(lab).any(axis=1))) + 1 if np.nan_to_num(lab).any() else 0
            add("shipped", str(w.condition), split, str(w.date), str(w.window_id), X, y, first,
                meta=dict(forecast_origin=str(w.forecast_origin), persistence=persistence(hv, vcut).reshape(-1),
                          tpl=None if tpl is None else tpl[tpl.window_id == str(w.window_id)], lmap=lmap))
    return rows


def _fit(X, y, w, seed):
    import lightgbm as lgb
    params = dict(objective="binary", learning_rate=0.05, num_leaves=63, min_data_in_leaf=200,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  seed=seed, deterministic=True, verbose=-1, num_threads=0)
    return lgb.train(params, lgb.Dataset(X, y, weight=w, feature_name=list(FEATURES), free_raw_data=True),
                     num_boost_round=300)


def _training_set(rows, weights):
    X = np.vstack([r["X"] for r in rows])
    y = np.concatenate([r["y"] for r in rows])
    w = np.concatenate([np.full(len(r["y"]), weights[id(r)]) for r in rows])
    keep = np.isfinite(y)
    return X[keep], y[keep], w[keep]


def onset_weights(rows):
    """Reweight generated onset windows so the first-queue step matches the shipped train windows."""
    w = {id(r): 1.0 for r in rows}
    gen = [r for r in rows if r["kind"] == "generated" and r["condition"] == "queue_onset"]
    ship = [r for r in rows if r["kind"] == "shipped" and r["split"] == "train"
            and r["condition"] == "queue_onset" and r["first"] > 0]
    if not gen or len(ship) < 5:
        return w, None
    pg = np.bincount([r["first"] for r in gen], minlength=H + 1)[1:] + 0.5
    ps = np.bincount([r["first"] for r in ship], minlength=H + 1)[1:] + 0.5
    ratio = (ps / ps.sum()) / (pg / pg.sum())
    for r in gen:
        w[id(r)] = float(ratio[r["first"] - 1])
    return w, {"shipped_first_step_counts": (ps - 0.5).astype(int).tolist(),
               "generated_first_step_counts": (pg - 0.5).astype(int).tolist()}


def run(rows, folds: int, seed: int, log=print):
    """Cross-validate, pick the selection rule, refit on everything, predict the scored windows."""
    report, preds = {}, {}
    for cond in CONDITIONS:
        rc = [r for r in rows if r["condition"] == cond]
        train_rows = [r for r in rc if r["split"] == "train" and np.isfinite(r["y"]).any()]
        if not train_rows:
            log(f"  {cond}: no train windows, falling back to persistence")
            continue
        weights, wrep = onset_weights(train_rows) if cond == "queue_onset" else ({id(r): 1.0 for r in train_rows}, None)
        dates = sorted({r["date"] for r in train_rows})
        rs = np.random.default_rng(seed)
        fold_of = dict(zip(dates, rs.integers(0, folds, len(dates))))
        cv = {rule: {"generated": [], "shipped": []} for rule in RULES}
        pers = {"generated": [], "shipped": []}
        for f in range(folds):
            tr = [r for r in train_rows if fold_of[r["date"]] != f]
            te = [r for r in train_rows if fold_of[r["date"]] == f]
            if not tr or not te:
                continue
            model = _fit(*_training_set(tr, weights), seed)
            for r in te:
                p = model.predict(r["X"])
                for rule in RULES:
                    cv[rule][r["kind"]].append(iou(select_cells(p, rule), r["y"]))
                if r["kind"] == "shipped":
                    pers["shipped"].append(iou(r["meta"]["persistence"] > 0, r["y"]))
            log(f"  {cond}: fold {f + 1}/{folds} done")

        def mean(v):
            return float(np.mean(v)) if v else float("nan")
        table = {f"{rule[0]}{'' if rule[1] is None else '@' + str(rule[1])}":
                 {k: mean(v) for k, v in cv[rule].items()} for rule in RULES}
        key = "shipped" if len(cv[RULES[0]]["shipped"]) >= 10 else "generated"
        best = max(RULES, key=lambda rl: np.nanmean([mean(cv[rl][key]), mean(cv[rl]["generated"])]))
        report[cond] = {"cv_iou_by_rule": table, "chosen_rule": list(best),
                        "persistence_iou_on_shipped_train_windows": mean(pers["shipped"]),
                        "n_generated_windows": sum(r["kind"] == "generated" for r in train_rows),
                        "n_shipped_train_windows": sum(r["kind"] == "shipped" for r in train_rows),
                        "onset_reweighting": wrep}
        log(f"  {cond}: chosen rule {best}, CV IoU shipped {mean(cv[best]['shipped']):.4f} "
            f"generated {mean(cv[best]['generated']):.4f}, persistence {mean(pers['shipped']):.4f}")
        model = _fit(*_training_set(train_rows, weights), seed)
        imp = dict(zip(FEATURES, model.feature_importance("gain").round(1).tolist()))
        report[cond]["top_features"] = dict(sorted(imp.items(), key=lambda kv: -kv[1])[:12])
        for r in rc:
            if r["split"] != "train":
                preds[r["window_id"]] = select_cells(model.predict(r["X"]), best)
    return report, preds


def submission_rows(rows, preds, split):
    """Fill each scored window's template; persistence wherever no model prediction exists."""
    out = []
    for r in rows:
        if r["kind"] != "shipped" or r["split"] != split or r["meta"]["tpl"] is None:
            continue
        t = r["meta"]["tpl"][["window_id", "timestamp", "link_id"]].copy()
        p = preds.get(r["window_id"], r["meta"]["persistence"] > 0)
        L = len(r["meta"]["lmap"])
        origin = pd.Timestamp(r["meta"]["forecast_origin"])
        h = ((pd.to_datetime(t.timestamp, utc=True) - origin) / pd.Timedelta(minutes=5)).round().astype(int).to_numpy()
        li = t.link_id.map(r["meta"]["lmap"])
        ok = (h >= 1) & (h <= H) & li.notna().to_numpy()
        idx = (np.clip(h, 1, H) - 1) * L + li.fillna(0).astype(int).to_numpy()
        t["queue_pred"] = np.where(ok, p[idx], 0).astype(int)
        out.append(t)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=["window_id", "timestamp", "link_id", "queue_pred"])


In [ ]:
%%writefile /kaggle/working/tfb/src/task4_odme.py
"""Task 4: recover path flows from link counts (origin-destination matrix estimation).

Solved on each split's own released link counts, with the split's own weak prior as a ridge anchor:

    min_f  ||A f - c||^2 + lam * ||f - b||^2,   f >= 0

A is the path-link incidence restricted to links that carry a released count (an unobserved connector is
not a measured zero). lam trades count fit (S_link) against staying near the prior (S_dev); the organizers'
baseline uses 0.05. Only S_link can be scored locally, so it is reported for each lambda tried.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import nnls


def operator(network: Path):
    paths = pd.read_csv(network / "path_set.csv", dtype={"path_id": str})
    inc = pd.read_csv(network / "path_link_incidence.csv", dtype={"path_id": str, "link_id": str})
    path_ids = paths.path_id.tolist()
    link_ids = inc.link_id.drop_duplicates().tolist()
    pi = {x: i for i, x in enumerate(path_ids)}
    li = {x: i for i, x in enumerate(link_ids)}
    A = np.zeros((len(link_ids), len(path_ids)))
    keep = inc.path_id.isin(pi)
    A[inc.link_id[keep].map(li).to_numpy(), inc.path_id[keep].map(pi).to_numpy()] = 1.0
    return path_ids, link_ids, A, paths


def solve(A, c, b, lam):
    aa = np.vstack([A, np.sqrt(lam) * np.eye(A.shape[1])])
    bb = np.concatenate([c, np.sqrt(lam) * b])
    f, _ = nnls(aa, bb, maxiter=50 * A.shape[1])
    return f


def s_link(A, f, c):
    return max(0.0, 1 - np.abs(A @ f - c).sum() / max(c.sum(), 1e-9))


def run_panel(release: Path, panel: str, split: str, lam: float, lams_report=(0.0, 0.01, 0.05, 0.2, 1.0)):
    base = release / "task4" / panel / split
    if not (base / "synthetic_link_counts.csv").exists():
        return None, None
    path_ids, link_ids, A, paths = operator(release / "corridors" / panel / "network")
    counts = pd.read_csv(base / "synthetic_link_counts.csv", dtype={"link_id": str})
    counts = counts.groupby("link_id")["count"].mean()
    measured = np.array([x in counts.index for x in link_ids])
    c = counts.reindex(np.asarray(link_ids)[measured]).to_numpy(float)
    prior = pd.read_csv(base / "synthetic_weak_prior.csv", dtype={"path_id": str}) \
        if (base / "synthetic_weak_prior.csv").exists() else pd.DataFrame(columns=["path_id", "path_flow"])
    b = pd.to_numeric(prior.drop_duplicates("path_id").set_index("path_id").path_flow, errors="coerce") \
        .reindex(path_ids).fillna(0.0).to_numpy(float)
    Am = A[measured]
    report = {"n_paths": len(path_ids), "n_count_links": int(measured.sum()), "lambda": lam,
              "S_link_prior_only": s_link(Am, b, c),
              "S_link_by_lambda": {str(l): s_link(Am, solve(Am, c, b, l), c) for l in lams_report}}
    f = solve(Am, c, b, lam)
    report["S_link"] = s_link(Am, f, c)
    report["l1_move_from_prior"] = float(np.abs(f - b).sum())

    tpl_path = base / "sample_submission_path_flow.csv"
    if tpl_path.exists():
        out = pd.read_csv(tpl_path, dtype=str)
    else:
        out = paths[["path_id", "origin_zone", "destination_zone"]].astype(str).copy()
        out["departure_time"] = str(prior.departure_time.iloc[0]) if "departure_time" in prior and len(prior) else "PM"
    out["panel"] = panel
    flow = dict(zip(path_ids, f))
    out["path_flow"] = out.path_id.map(flow).fillna(0.0).clip(lower=0).round(4)
    cols = [c_ for c_ in ("panel", "departure_time", "path_id", "origin_zone", "destination_zone", "path_flow")
            if c_ in out.columns]
    return out[cols], report


In [ ]:
%%writefile /kaggle/working/tfb/src/pipeline.py
"""One-command TrafficFlowBench pipeline: data -> Task 1, 2, 4 -> Kaggle upload file + report.

    python src/pipeline.py                         # data already unpacked under data/kaggle_public
    python src/pipeline.py --download              # fetch from Kaggle first (needs a Kaggle API token)
    python src/pipeline.py --data-url gs://bucket/2026-ieee-big-data-traffic-flow-bench.zip
    python src/pipeline.py --panel D12_I5_N        # one corridor, for a quick run

Task 3 has no output of its own: it is scored on the Task 1 rows.
"""
from __future__ import annotations

import argparse
import json
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))

import task1_state as t1  # noqa: E402
import task2_queue as t2  # noqa: E402
import task4_odme as t4  # noqa: E402
from common import CACHE, DEFAULT_RELEASE, OUTPUTS, ROOT, cached, network, panels, weekday_of  # noqa: E402

SLUG = "2026-ieee-big-data-traffic-flow-bench"
SCORED = ("validation", "private")
T0 = time.time()


def log(msg=""):
    print(f"[{time.time() - T0:7.0f}s] {msg}", flush=True)


def find_release(root: Path) -> Path | None:
    if not root.exists():
        return None
    for cand in [root, *sorted(p for p in root.glob("*/") if p.is_dir()), *sorted(p for p in root.glob("*/*/") if p.is_dir())]:
        if (cand / "corridors").is_dir() and (cand / "task1").is_dir():
            return cand
    return None


def acquire(args) -> Path:
    data = ROOT / "data"
    data.mkdir(exist_ok=True)
    archive = None
    if args.download:
        import os
        for alias in ("Kaggle_competition", "KAGGLE_COMPETITION"):
            if not os.environ.get("KAGGLE_API_TOKEN") and os.environ.get(alias):
                os.environ["KAGGLE_API_TOKEN"] = os.environ[alias]
        log("Downloading from Kaggle (about 13 GB unpacked)")
        subprocess.run(["kaggle", "competitions", "download", "-c", SLUG, "-p", str(data)], check=True)
        archive = data / f"{SLUG}.zip"
    elif args.data_url:
        archive = data / Path(args.data_url.split("?")[0]).name
        log(f"Fetching {args.data_url}")
        if args.data_url.startswith("gs://"):
            subprocess.run(["gsutil", "cp", args.data_url, str(archive)], check=True)
        else:
            subprocess.run(["curl", "-fL", "-o", str(archive), args.data_url], check=True)
    if archive is not None and archive.exists():
        log(f"Unpacking {archive.name}")
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(data)
        if not args.keep_archive:
            archive.unlink()
    rel = find_release(Path(args.release_root)) if args.release_root else find_release(data)
    if rel is None or not (Path(rel) / "corridors").is_dir():
        raise SystemExit(
            f"No release found under {data}. Unpack the Kaggle data so that data/kaggle_public/corridors exists, "
            "or pass --release-root, --download or --data-url.")
    return Path(rel).resolve()


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--release-root", type=Path)
    ap.add_argument("--download", action="store_true", help="download the data with the Kaggle CLI")
    ap.add_argument("--data-url", help="gs:// or https:// URL of the competition zip")
    ap.add_argument("--keep-archive", action="store_true")
    ap.add_argument("--panel", action="append", help="limit to these corridors (repeatable)")
    ap.add_argument("--max-windows", type=int, default=600,
                    help="generated Task 2 training windows per corridor and condition")
    ap.add_argument("--folds", type=int, default=4)
    ap.add_argument("--lam", type=float, default=0.05, help="Task 4 prior regularization")
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--rebuild-cache", action="store_true")
    ap.add_argument("--output-dir", type=Path, default=OUTPUTS)
    args = ap.parse_args()

    rel = acquire(args)
    out = args.output_dir.resolve()
    out.mkdir(parents=True, exist_ok=True)
    all_panels = panels(rel)
    sel = [p for p in all_panels if not args.panel or p in args.panel]
    log(f"Release {rel}; corridors: {', '.join(sel)}")
    rng = np.random.default_rng(args.seed)
    report = {"release_root": str(rel), "panels": sel, "task1": {}, "task2": {}, "task4": {}}
    state_path, odme_parts, t2_rows = out / "state_submission.csv", [], []
    if state_path.exists():
        state_path.unlink()
    first_state = True

    for panel in sel:
        log(f"== {panel}")
        z_tr = cached(rel, panel, "train", args.rebuild_cache)
        scored = {}
        for s in SCORED:
            if (rel / "corridors" / panel / s / "mainline_states_masked").exists():
                scored[s] = cached(rel, panel, s, args.rebuild_cache)
        net = network(rel, panel, z_tr["links"])
        lanes, vmax = net.lanes.to_numpy(float), 1.3 * net.free_speed_kmh.to_numpy(float) + 10
        log(f"  {len(z_tr['links'])} links, train {len(z_tr['dates'])} days, "
            f"{int(z_tr['target'].sum()):,} train targets")

        # Task 1: profiles from released cells of every split, method chosen on train
        arrays_v = [z_tr["speed"]] + [z["speed"] for z in scored.values()]
        arrays_q = [z_tr["flow"]] + [z["flow"] for z in scored.values()]
        wds = [weekday_of(z_tr["dates"])] + [weekday_of(z["dates"]) for z in scored.values()]
        P_v, P_q = t1.profile(arrays_v, wds), t1.profile(arrays_q, wds)
        log("  Task 1 candidates on train:")
        sel1 = t1.select(z_tr, P_v, P_q, lanes, vmax, log=log)
        cfg_v = (sel1["speed_method"]["tau"], sel1["speed_method"]["nbr"])
        cfg_q = (sel1["flow_method"]["tau"], sel1["flow_method"]["nbr"])
        log(f"  Task 1 train S_state {sel1['train_score']['S_state']:.4f} "
            f"(profile only {sel1['train_score_profile_only']['S_state']:.4f}); "
            f"speed: {sel1['speed_method']['method']}, flow: {sel1['flow_method']['method']}")
        report["task1"][panel] = sel1
        for s, z in scored.items():
            tpl = pd.read_csv(rel / "task1" / panel / s / "sample_submission_state.csv", dtype=str)
            pv, pq = t1.predict(z, P_v, P_q, cfg_v, cfg_q, vmax)
            rows = t1.submission_rows(panel, tpl, z, pv, pq)
            rows.to_csv(state_path, mode="w" if first_state else "a", header=first_state, index=False)
            first_state = False
            log(f"  Task 1 {s}: {len(rows):,} rows")

        # Task 2 tables (trained jointly across corridors below)
        if (rel / "task2" / panel).exists():
            rows2 = t2.panel_tables(rel, panel, z_tr, scored, net, rng, args.max_windows)
            t2_rows += rows2
            log(f"  Task 2: {sum(r['kind'] == 'generated' for r in rows2)} generated windows, "
                f"{sum(r['kind'] == 'shipped' for r in rows2)} published windows")

        # Task 4
        for s in SCORED:
            sub, rep = t4.run_panel(rel, panel, s, args.lam)
            if sub is not None:
                odme_parts.append(sub)
                report["task4"][f"{panel}/{s}"] = rep
                log(f"  Task 4 {s}: S_link {rep['S_link']:.4f} (prior only {rep['S_link_prior_only']:.4f})")
        del z_tr, scored

    log("== Task 2 models")
    rep2, preds = t2.run(t2_rows, args.folds, args.seed, log=log)
    report["task2"] = rep2
    queue = pd.concat([t2.submission_rows(t2_rows, preds, s) for s in SCORED], ignore_index=True)
    queue.to_csv(out / "queue_submission.csv", index=False)
    log(f"  Task 2: {len(queue):,} rows, {int(queue.queue_pred.sum()):,} predicted queued")
    pd.concat(odme_parts, ignore_index=True).to_csv(out / "odme_submission.csv", index=False)

    key = rel / "submission_key.csv"
    if key.exists():
        log("== Merging into the upload file")
        subprocess.run([sys.executable, str(ROOT / "official" / "src" / "merge_submissions.py"),
                        "--state", str(state_path), "--queue", str(out / "queue_submission.csv"),
                        "--odme", str(out / "odme_submission.csv"), "--key", str(key),
                        "--output", str(out / "submission.csv")], check=True)
        with zipfile.ZipFile(out / "submission.csv.zip", "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(out / "submission.csv", "submission.csv")
    else:
        log("No submission_key.csv in the release; per-task files written, merge skipped")

    report["summary"] = summarize(report)
    (out / "report.json").write_text(json.dumps(report, indent=1, default=float))
    plots(report, out)
    log(f"Done. Outputs in {out}")
    print(json.dumps(report["summary"], indent=1, default=float))


def summarize(report):
    s1 = [v["train_score"]["S_state"] for v in report["task1"].values()]
    s1b = [v["train_score_profile_only"]["S_state"] for v in report["task1"].values()]
    q = {}
    for cond, r in report["task2"].items():
        rule = r["chosen_rule"]
        name = f"{rule[0]}{'' if rule[1] is None else '@' + str(rule[1])}"
        q[cond] = {"model_cv_iou_shipped": r["cv_iou_by_rule"][name]["shipped"],
                   "model_cv_iou_generated": r["cv_iou_by_rule"][name]["generated"],
                   "persistence_iou_shipped": r["persistence_iou_on_shipped_train_windows"]}
    sl = [v["S_link"] for v in report["task4"].values()]
    return {"task1_train_S_state_mean": float(np.mean(s1)) if s1 else None,
            "task1_train_S_state_profile_baseline": float(np.mean(s1b)) if s1b else None,
            "task2_cv": q,
            "task4_S_link_mean": float(np.mean(sl)) if sl else None}


def plots(report, out):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    if report["task1"]:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
        for panel, v in report["task1"].items():
            c = pd.DataFrame(v["candidates"])
            ax[0].plot(c.method, c.rmse_speed, marker="o", label=panel)
            ax[1].plot(c.method, c.rmse_flow_per_lane, marker="o", label=panel)
        for a, t in zip(ax, ("Speed RMSE (km/h)", "Flow RMSE (veh/h/lane)")):
            a.set_title(t)
            a.tick_params(axis="x", rotation=60, labelsize=7)
        ax[1].legend(fontsize=7)
        fig.suptitle("Task 1 candidates on train target cells (lower is better)")
        fig.tight_layout()
        fig.savefig(out / "task1_methods.png", dpi=120)
        plt.close(fig)
    if report["task2"]:
        conds = list(report["task2"])
        model = [report["summary"]["task2_cv"][c]["model_cv_iou_shipped"] for c in conds]
        pers = [report["summary"]["task2_cv"][c]["persistence_iou_shipped"] for c in conds]
        fig, ax = plt.subplots(figsize=(6, 4))
        x = np.arange(len(conds))
        ax.bar(x - 0.2, pers, 0.4, label="persistence baseline")
        ax.bar(x + 0.2, model, 0.4, label="LightGBM")
        ax.set_xticks(x, conds)
        ax.set_ylabel("IoU, organizers' train windows (CV)")
        ax.legend()
        fig.tight_layout()
        fig.savefig(out / "task2_iou.png", dpi=120)
        plt.close(fig)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tfb/official/src/merge_submissions.py
"""Build the single upload file from one file per task.

The leaderboard reads six columns keyed by the organizer's ``submission_id``:

    submission_id,task,speed_kmh,flow_vph,queue_pred,path_flow

Working in that shape is awkward, so most people keep one file per task in its
natural key. This script joins those files onto ``submission_key.csv`` and
writes the upload file, in template order, with a value in every cell.

Task 3 has no file. It is scored on the Task 1 state rows.

    python src/merge_submissions.py \
      --state state_submission.csv \
      --queue queue_submission.csv \
      --odme  path_flow_submission.csv \
      --key   submission_key.csv \
      --output submission.csv

The key file is large, so it is streamed rather than held in memory.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import pandas as pd

CHUNK = 1_000_000
OUT_COLUMNS = ["submission_id", "task", "speed_kmh", "flow_vph", "queue_pred", "path_flow"]
# The natural key of each per-task file, and the values it supplies.
KEYS = {"state": ["panel", "timestamp", "station_id", "link_id", "mask_regime"],
        "queue": ["window_id", "timestamp", "link_id"],
        "odme": ["panel", "departure_time", "path_id"]}
VALUES = {"state": ["speed_kmh", "flow_vph"], "queue": ["queue_pred"], "odme": ["path_flow"]}


def load(path: Path, task: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{task} submission not found: {path}")
    frame = pd.read_csv(path)
    missing = sorted(set(KEYS[task] + VALUES[task]) - set(frame.columns))
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")
    frame = frame[KEYS[task] + VALUES[task]].copy()
    for column in KEYS[task]:
        frame[column] = (pd.to_datetime(frame[column], utc=True) if column == "timestamp"
                         else frame[column].astype(str))
    for column in VALUES[task]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    duplicated = int(frame.duplicated(KEYS[task]).sum())
    if duplicated:
        raise ValueError(f"{path} has {duplicated} duplicate rows over {KEYS[task]}")
    return frame.set_index(KEYS[task])


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--state", type=Path, required=True)
    ap.add_argument("--queue", type=Path, required=True)
    ap.add_argument("--odme", type=Path, required=True)
    ap.add_argument("--key", type=Path, required=True,
                    help="submission_key.csv, from the same folder as sample_submission.csv")
    ap.add_argument("--output", type=Path, required=True)
    args = ap.parse_args()

    tasks = {"state": load(args.state.resolve(), "state"),
             "queue": load(args.queue.resolve(), "queue"),
             "odme": load(args.odme.resolve(), "odme")}
    args.output.parent.mkdir(parents=True, exist_ok=True)
    written, gaps, first = 0, {t: 0 for t in tasks}, True
    for chunk in pd.read_csv(args.key.resolve(), chunksize=CHUNK, dtype=str):
        chunk["timestamp"] = pd.to_datetime(chunk.timestamp, utc=True, errors="coerce")
        out = pd.DataFrame({"submission_id": chunk.submission_id.astype("int64"),
                            "task": chunk.task.astype(str)})
        for column in ("speed_kmh", "flow_vph", "queue_pred", "path_flow"):
            out[column] = 0.0
        for task, table in tasks.items():
            rows = out.task == task
            if not rows.any():
                continue
            wanted = pd.MultiIndex.from_frame(chunk.loc[rows, KEYS[task]])
            found = table.reindex(wanted)
            for column in VALUES[task]:
                values = found[column].to_numpy()
                gaps[task] += int(pd.isna(values).sum())
                out.loc[rows, column] = pd.Series(values).fillna(0.0).to_numpy()
        out[OUT_COLUMNS].to_csv(args.output.resolve(), mode="w" if first else "a",
                                header=first, index=False)
        first = False
        written += len(out)
        print(f"  {written:,} rows", end="\r", flush=True)

    print(f"Wrote {written:,} rows to {args.output.resolve()}")
    for task, n in gaps.items():
        if n:
            print(f"WARNING: {n:,} {task} values had no match and were written as 0. "
                  f"Check the {KEYS[task]} in your {task} file.")
    if not any(gaps.values()):
        print("Every scored cell was filled from your files.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tfb/official/config/corridors.json
{
  "release_name": "TrafficFlowBench Five-Corridor Public Release",
  "families": [
    {
      "family_id": "D7_I10",
      "district": 7,
      "freeway": "I-10"
    },
    {
      "family_id": "D7_I210",
      "district": 7,
      "freeway": "I-210"
    },
    {
      "family_id": "D7_I405",
      "district": 7,
      "freeway": "I-405"
    },
    {
      "family_id": "D12_I5",
      "district": 12,
      "freeway": "I-5"
    },
    {
      "family_id": "D12_I405",
      "district": 12,
      "freeway": "I-405"
    }
  ],
  "panels": [
    {
      "corridor_id": "D7_I10_E",
      "family_id": "D7_I10",
      "district": 7,
      "freeway_number": 10,
      "direction": "E"
    },
    {
      "corridor_id": "D7_I10_W",
      "family_id": "D7_I10",
      "district": 7,
      "freeway_number": 10,
      "direction": "W"
    },
    {
      "corridor_id": "D7_I210_E",
      "family_id": "D7_I210",
      "district": 7,
      "freeway_number": 210,
      "direction": "E"
    },
    {
      "corridor_id": "D7_I210_W",
      "family_id": "D7_I210",
      "district": 7,
      "freeway_number": 210,
      "direction": "W"
    },
    {
      "corridor_id": "D7_I405_N",
      "family_id": "D7_I405",
      "district": 7,
      "freeway_number": 405,
      "direction": "N"
    },
    {
      "corridor_id": "D7_I405_S",
      "family_id": "D7_I405",
      "district": 7,
      "freeway_number": 405,
      "direction": "S"
    },
    {
      "corridor_id": "D12_I5_N",
      "family_id": "D12_I5",
      "district": 12,
      "freeway_number": 5,
      "direction": "N"
    },
    {
      "corridor_id": "D12_I5_S",
      "family_id": "D12_I5",
      "district": 12,
      "freeway_number": 5,
      "direction": "S"
    },
    {
      "corridor_id": "D12_I405_N",
      "family_id": "D12_I405",
      "district": 12,
      "freeway_number": 405,
      "direction": "N"
    },
    {
      "corridor_id": "D12_I405_S",
      "family_id": "D12_I405",
      "district": 12,
      "freeway_number": 405,
      "direction": "S"
    }
  ]
}


In [ ]:
import glob, os
rel = sorted(glob.glob('/kaggle/input/**/corridors', recursive=True))[0].rsplit('/corridors', 1)[0]
os.environ['TFB_CACHE'] = '/kaggle/temp/cache'
print(rel)

In [ ]:
!cd /kaggle/working/tfb && TFB_CACHE=/kaggle/temp/cache python src/pipeline.py --release-root "{rel}" --output-dir /kaggle/working/outputs

In [ ]:
import pandas as pd
print(pd.read_json('/kaggle/working/outputs/report.json', typ='series')['summary'])